Value-Based RL

Q-Learning

Forget policies for a moment.

Instead of asking:

"What probability should I assign to each action?"

we ask:

"How valuable is taking action a in state s?"

That's:

Q(s,a)

Example:

State = robot at intersection

Q(state, left)  = 3.2
Q(state, right) = 7.8
Q(state, forward) = 5.1

The agent can simply choose:

argmax Q(s,a)

→ right.

1. Where does Q come from?

We don't know the correct Q-values initially.

Start with garbage:

Q:

        LEFT    RIGHT
S0      0       0
S1      0       0
...

Then interact with the environment.

Suppose:

S0
 ↓ RIGHT
reward = +1
 ↓
S1

We want:

Q(S0,RIGHT)

to reflect:

immediate reward + future value.

The Q-learning update is:

Q(s,a)←Q(s,a)+α[r+γ
a
′
max
	​

Q(s
′
,a
′
)−Q(s,a)]
	​


Don't memorize the formula yet.

The important part is:

current Q
      ↓
compare against
      ↓
reward + best estimated future
      ↓
move current Q toward that target
2. This is another TD method

Look at the target:

r+γ
a
′
max
	​

Q(s
′
,a
′
)

We're doing the same basic trick as TD learning:

Use the estimated future instead of waiting for the whole episode.

But now we're estimating action values rather than state values.

That's the important connection.

3. Why max?

Suppose after reaching S1:

Q(S1, LEFT)   = 4
Q(S1, RIGHT)  = 8
Q(S1, FORWARD) = 3

Then the best future value is:

8

So:

a
max
	​

Q(S1,a)=8

We're effectively asking:

"If I reach this next state, what's the best future I can get?"

That's why Q-learning is naturally value-based.

4. Exploration

There's an immediate problem.

If we always do:

action = argmax(Q[state])

then initially all values are identical.

The agent might pick one action forever and never discover something better.

So we use ε-greedy:

with probability ε:
    random action

otherwise:
    best Q action

Example:

ε = 0.2

means roughly:

20% → explore
80% → exploit

Usually we decay ε:

early training:
ε = 1.0

later:
ε = 0.05

So the agent explores heavily at first and becomes increasingly greedy.

Environment:

S . . .
. . X .
. . . .
. . . G

Where:

S = start
G = goal
X = obstacle

Actions:

0 = up
1 = down
2 = left
3 = right

Reward:

goal      → +10
normal    → -0.1
obstacle  → -1

The agent has to discover the path.

In [5]:
import random
import numpy as np


GRID_SIZE = 4
ACTIONS = 4

ALPHA = 0.1
GAMMA = 0.99

EPISODES = 5000

EPSILON = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.995


q_table = np.zeros(
    (GRID_SIZE, GRID_SIZE, ACTIONS),
    dtype=np.float32,
)


START = (0, 0)
GOAL = (3, 3)
OBSTACLE = (1, 2)


def reset():
    return START


def step(state, action):

    row, col = state

    next_row = row
    next_col = col

    if action == 0:
        next_row -= 1
    elif action == 1:
        next_row += 1
    elif action == 2:
        next_col -= 1
    elif action == 3:
        next_col += 1

    # Boundary
    if not (
        0 <= next_row < GRID_SIZE
        and 0 <= next_col < GRID_SIZE
    ):
        return state, -1, False

    next_state = (next_row, next_col)

    # Obstacle
    if next_state == OBSTACLE:
        return state, -1, False

    # Goal
    if next_state == GOAL:
        return next_state, 10, True

    return next_state, -0.1, False


def choose_action(state, epsilon):

    if random.random() < epsilon:
        return random.randrange(ACTIONS)

    row, col = state

    return int(
        np.argmax(q_table[row, col])
    )


epsilon = EPSILON


for episode in range(EPISODES):

    state = reset()

    for step_count in range(100):

        action = choose_action(
            state,
            epsilon,
        )

        next_state, reward, done = step(
            state,
            action,
        )

        row, col = state

        next_row, next_col = next_state

        current_q = q_table[
            row,
            col,
            action,
        ]

        if done:

            target = reward

        else:

            target = (
                reward
                + GAMMA
                * np.max(
                    q_table[
                        next_row,
                        next_col,
                    ]
                )
            )

        q_table[
            row,
            col,
            action,
        ] = current_q + ALPHA * (
            target - current_q
        )

        state = next_state

        if done:
            break

    epsilon = max(
        EPSILON_MIN,
        epsilon * EPSILON_DECAY,
    )


print("\nLearned policy:\n")

symbols = {
    0: "↑",
    1: "↓",
    2: "←",
    3: "→",
}

for row in range(GRID_SIZE):

    line = ""

    for col in range(GRID_SIZE):

        if (row, col) == START:
            line += " S "

        elif (row, col) == GOAL:
            line += " G "

        elif (row, col) == OBSTACLE:
            line += " X "

        else:

            action = np.argmax(
                q_table[row, col]
            )

            line += f" {symbols[action]} "

    print(line)


Learned policy:

 S  →  →  ↓ 
 →  ↓  X  ↓ 
 →  →  →  ↓ 
 →  →  →  G 


Gamma (γ) controls how much the agent cares about future rewards.

In our Q-learning:

Q←r+γmaxQ(s
′
,a
′
)

We used:

γ = 0.99
Effect
γ ≈ 0 → care mostly about immediate reward.
γ ≈ 1 → care strongly about long-term reward.

So in our grid:

γ = 0.1
→ "Is this next move good?"

γ = 0.99
→ "Will this move eventually get me to the goal?"

That's why with high γ, the agent can learn a longer path toward the +10 goal despite the small -0.1 step penalties.